# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Each record set and its fields are identified by their `@id` values. We'll list them for exploration.

In [ ]:
# Get available record sets from metadata
record_sets = dataset.metadata.recordSet
print("Available Record Sets:")
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {rs['@id']}    Name: {rs.get('name', rs['@id'])}")
    if 'field' in rs:
        fields = rs['field']
        print("    Fields:")
        for f in fields:
            print(f"        - {f['@id']} (Name: {f.get('name', f['@id'])}, DataType: {f.get('dataType', 'unknown')})")
    if 'column' in rs:
        columns = rs['column']
        print("    Columns:")
        for col in columns:
            print(f"        - {col['@id']} (Name: {col.get('name', col['@id'])})")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We'll use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
df_dict = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
print("Extracting data for each RecordSet @id:")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df_dict[record_set_id] = pd.DataFrame(records)
    print(f"- {record_set_id}: rows={len(records)}, columns={df_dict[record_set_id].columns.tolist()}")
if record_set_ids:
    # Show columns and preview for the first record set
    main_record_set_id = record_set_ids[0]
    print(f"\nFirst RecordSet @id: {main_record_set_id}")
    print("Columns:", df_dict[main_record_set_id].columns.tolist())
    df_dict[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll select fields using their `@id` from the overview above.

In [ ]:
# Choose the main RecordSet for EDA
main_record_set_id = record_set_ids[0]
df = df_dict[main_record_set_id].copy()

# Identify numeric fields based on metadata
numeric_field_id = None
group_field_id = None
fields_info = [f for rs in dataset.metadata.recordSet if rs['@id']==main_record_set_id for f in rs.get('field',[])]
for f in fields_info:
    dt = f.get('dataType','')
    if dt in ['schema:Float', 'schema:Integer', 'Integer', 'Float', 'Number'] and numeric_field_id is None:
        numeric_field_id = f['@id']
    if dt in ['schema:Text', 'Text'] and group_field_id is None:
        group_field_id = f['@id']
print(f"Numeric Field @id: {numeric_field_id}")
print(f"Group Field @id: {group_field_id}")

# Filtering, normalization, grouping (only if fields exist in data)
if numeric_field_id and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records: {numeric_field_id} > {threshold}")
    print(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print("Normalized:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships. All visualizations reference fields by their `@id` value, consistent with Croissant schema context.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot grouped by the group field
if group_field_id and numeric_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(F"{numeric_field_id} per {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load and understand FAIR^2 dataset metadata using `mlcroissant`.
- Identify record sets and fields via their `@id` values for robust referencing.
- Extract tabular data for each record set and perform basic EDA.
- Visualize key distributions and groupings using field `@id`s.

For further analysis or ML tasks, continue referencing dataset entities by their Croissant `@id` fields for consistency and clarity.